# Hirup PM2.5 nowcast: evaluation

Standard evaluation notebook (SPEC 16, level *Model*). It reads the artefacts `hirup-ml train` saved for one model version and plots them. Run it from the repository root after training:

```bash
uv run --with jupyter jupyter lab ml/notebooks/evaluation.ipynb
```

Only real station data may be reported from here. Do not point it at toy data.

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from hirup_ml import metrics, registry
from hirup_ml.config import models_dir

entry = registry.active() or (registry.entries() or [None])[-1]
assert entry is not None, 'No model registered yet: run `uv run hirup-ml train`'
folder = models_dir() / entry['model_version']
validation = json.loads((folder / 'validation.json').read_text())
oof = pd.read_parquet(folder / 'oof.parquet')
entry['model_version'], entry['is_active'], entry['promotion'], entry['data_window']

## 1. Model vs baselines, leave-one-station-out

Target (Tabel 4.1): RMSE below IDW, ideally by 10% or more.

In [ ]:
rows = {name: validation['loso'][name] for name in ('lightgbm', 'idw', 'cams')}
display(pd.DataFrame(rows).T)
print('RMSE improvement vs IDW:', validation['loso']['rmse_improvement_vs_idw'])
print('Spatial block CV:', validation['spatial_block_cv']['lightgbm'])
print('Temporal hold-out:', validation['temporal_holdout'])
print('Source hold-out:', validation['source_holdout'])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharex=True, sharey=True)
limit = float(np.nanpercentile(oof['y'], 99.5))
for ax, column, title in zip(axes, ('pred', 'idw'), ('LightGBM (LOSO)', 'IDW (LOSO)')):
    ax.scatter(oof['y'], oof[column], s=3, alpha=0.2)
    ax.plot([0, limit], [0, limit], color='black', linewidth=1)
    ax.set(title=f"{title}: RMSE {metrics.rmse(oof['y'].to_numpy(), oof[column].to_numpy()):.1f}", xlabel='Observed PM2.5 (ug/m3)', xlim=(0, limit), ylim=(0, limit))
axes[0].set_ylabel('Predicted PM2.5 (ug/m3)')
plt.tight_layout()

## 2. Per-station error

A model that only wins at a few stations is not robust.

In [ ]:
per_station = oof.groupby('station_id').apply(lambda g: pd.Series({
    'n': len(g),
    'rmse_model': metrics.rmse(g['y'].to_numpy(), g['pred'].to_numpy()),
    'rmse_idw': metrics.rmse(g['y'].to_numpy(), g['idw'].to_numpy()),
}), include_groups=False)
per_station['model_wins'] = per_station['rmse_model'] < per_station['rmse_idw']
print('Stations where the model beats IDW:', int(per_station['model_wins'].sum()), 'of', len(per_station))
per_station.sort_values('rmse_model')

## 3. Uncertainty: P10 to P90 coverage

Target (Tabel 4.1): 70% to 90%.

In [ ]:
display(pd.Series(validation['intervals']))
inside = (oof['y'] >= oof['q10']) & (oof['y'] <= oof['q90'])
hour = oof['ts'].dt.tz_convert('Asia/Jakarta').dt.hour
inside.groupby(hour).mean().plot(kind='bar', figsize=(10, 3), title='Raw P10-P90 coverage by local hour', ylim=(0, 1))
plt.axhline(0.8, color='black', linewidth=1);

## 4. Error analysis (SPEC 11.4)

By road class, local hour, ISPU category, and the 06:00 to 09:00 rush.

In [ ]:
def table(group):
    return pd.DataFrame({key: {'n': v['model']['n'], 'rmse_model': v['model']['rmse'], 'rmse_idw': v['idw']['rmse'], 'bias_model': v['model']['bias']} for key, v in validation['error_analysis'][group].items()}).T

for group in ('rush_hour', 'road_class', 'ispu_category'):
    display(table(group))
by_hour = table('local_hour')
by_hour.index = by_hour.index.astype(int)
by_hour.sort_index()[['rmse_model', 'rmse_idw']].plot(figsize=(10, 3), title='RMSE by local hour', xlabel='Hour (WIB)', ylabel='RMSE (ug/m3)');

## 5. Feature contributions (TreeSHAP)

In [ ]:
shap = pd.Series(json.loads((folder / 'shap_summary.json').read_text()))
shap.head(20)[::-1].plot(kind='barh', figsize=(8, 6), title='Mean |SHAP| (ug/m3), top 20 features');

## 6. Map sanity check

The latest published nowcast. Look for grid artefacts: stripes, blocks, or a seam along the CAMS cell boundary.

In [ ]:
import rasterio

from hirup_ml.config import manifests_dir
from hirup_pipelines.settings import data_dir

manifest = json.loads((manifests_dir() / 'nowcast.json').read_text())
fig, axes = plt.subplots(1, 3, figsize=(12, 7))
for ax, name in zip(axes, ('p10', 'p50', 'p90')):
    with rasterio.open(data_dir() / manifest['files'][name]) as src:
        image = ax.imshow(src.read(1), cmap='magma')
    ax.set_title(f"{name} {manifest['valid_hour']}
{manifest['model_version']} ({manifest['data_quality']})", fontsize=8)
    ax.axis('off')
    plt.colorbar(image, ax=ax, shrink=0.6)
manifest['metrics']